In [ ]:
# Purpose:
#   Address the remaining realised-validation limitation:
#   the previous analysis used only one fixed 12-case cohort.
#
# Design:
#   - 2 additional non-overlapping 12-case cohorts
#   - fixed-seed random sampling
#   - temporally held-out final test cohort only
#   - sampling independent of predictions and realised outcomes
#   - 3 predictive models
#   - lambda = 0.5 only
#
# Total new MILP runs:
#   2 cohorts × 3 models = 6
#
# IMPORTANT:
#   Actual surgical duration is NEVER used during optimisation.
#   It is introduced only during ex-post realised replay.
# ============================================================

import os
import time
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Load frozen final MILP engine
# ------------------------------------------------------------

%run milp_engine.ipynb

print("Final MILP engine loaded.")

In [ ]:
# ============================================================
# 1. FROZEN EXPERIMENTAL CONFIGURATION
# ============================================================

RANDOM_SEED = 42

N_CASES = 12

N_ADDITIONAL_COHORTS = 2

VALIDATION_LAMBDA = 0.5


# ------------------------------------------------------------
# Final MILP settings
# ------------------------------------------------------------

N_ROOMS = 3

ROOM_CAPACITY = 480

TURNOVER = 20

BALANCE_WEIGHT = 0.10

DURATION_CAP = 360.0

TIME_LIMIT = 300


# ------------------------------------------------------------
# Input files
# ------------------------------------------------------------

ORIGINAL_COHORT_PATH = (
    "Cohort/fixed_cohort.csv"
)

ACTUAL_TEST_PATH = (
    "new_data/mover_epic_final_test_features.csv"
)


MODELS = {

    "LightGBM":
        "Prescriptive/LightGBM_prescriptive_optimizer_inputs.csv",

    "RF":
        "Prescriptive/RF_prescriptive_optimizer_inputs.csv",

    "XGBoost":
        "Prescriptive/XGBoost_prescriptive_optimizer_inputs.csv"
}


# ------------------------------------------------------------
# Output directories
# ------------------------------------------------------------

RESULTS_DIR = "Results"

COHORT_DIR = "Cohort"

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)

os.makedirs(
    COHORT_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# Frozen cohort files
# ------------------------------------------------------------

COHORT_A_PATH = os.path.join(
    COHORT_DIR,
    "additional_cohort_A.csv"
)

COHORT_B_PATH = os.path.join(
    COHORT_DIR,
    "additional_cohort_B.csv"
)


# ------------------------------------------------------------
# Validation output files
# ------------------------------------------------------------

SCHEDULE_OUTPUT = os.path.join(
    RESULTS_DIR,
    "additional_cohort_validation_schedules.csv"
)

ROOM_OUTPUT = os.path.join(
    RESULTS_DIR,
    "additional_cohort_validation_room_results.csv"
)

SUMMARY_OUTPUT = os.path.join(
    RESULTS_DIR,
    "additional_cohort_validation_summary.csv"
)

CHECKPOINT_OUTPUT = os.path.join(
    RESULTS_DIR,
    "additional_cohort_validation_checkpoint.csv"
)


print("=" * 75)
print("STEP 4D — ADDITIONAL COHORT VALIDATION")
print("=" * 75)

print(f"Random seed           : {RANDOM_SEED}")
print(f"Additional cohorts    : {N_ADDITIONAL_COHORTS}")
print(f"Cases per cohort      : {N_CASES}")
print(f"Validation lambda     : {VALIDATION_LAMBDA}")
print(f"Models                : {list(MODELS.keys())}")
print(f"Rooms                 : {N_ROOMS}")
print(f"Room capacity         : {ROOM_CAPACITY} min")
print(f"Turnover              : {TURNOVER} min")
print(f"Balance weight beta   : {BALANCE_WEIGHT}")
print(f"Duration cap          : {DURATION_CAP} min")
print(f"Solver time limit     : {TIME_LIMIT} s")

print(
    f"New MILP runs         : "
    f"{N_ADDITIONAL_COHORTS * len(MODELS)}"
)

In [ ]:
# ============================================================
# 2. LOAD ORIGINAL COHORT + FINAL HELD-OUT ACTUAL DATA
# ============================================================

assert os.path.exists(
    ORIGINAL_COHORT_PATH
), (
    f"Original cohort not found: "
    f"{ORIGINAL_COHORT_PATH}"
)


assert os.path.exists(
    ACTUAL_TEST_PATH
), (
    f"Final test data not found: "
    f"{ACTUAL_TEST_PATH}"
)


# ------------------------------------------------------------
# Original fixed cohort
# ------------------------------------------------------------

original_cohort = pd.read_csv(
    ORIGINAL_COHORT_PATH
)


assert "LOG_ID" in original_cohort.columns

assert len(original_cohort) == N_CASES

assert original_cohort["LOG_ID"].notna().all()

assert original_cohort["LOG_ID"].is_unique


original_ids = (
    original_cohort["LOG_ID"]
    .astype(str)
    .tolist()
)


# ------------------------------------------------------------
# Final temporally held-out test cohort
# ------------------------------------------------------------

actual_df = pd.read_csv(
    ACTUAL_TEST_PATH
)


required_actual_columns = {
    "LOG_ID",
    "ACTUAL_DURATION"
}


missing_actual_columns = (
    required_actual_columns
    -
    set(actual_df.columns)
)


assert not missing_actual_columns, (
    f"Missing required actual-data columns: "
    f"{missing_actual_columns}"
)


# ------------------------------------------------------------
# Final pipeline integrity
# ------------------------------------------------------------

assert len(actual_df) == 17083, (
    f"Expected final test cohort = 17083, "
    f"found {len(actual_df)}."
)


actual_df = actual_df.copy()

actual_df["LOG_ID"] = (
    actual_df["LOG_ID"]
    .astype(str)
)


assert actual_df["LOG_ID"].notna().all()

assert actual_df["LOG_ID"].is_unique

assert actual_df["ACTUAL_DURATION"].notna().all()

assert np.isfinite(
    actual_df["ACTUAL_DURATION"]
    .to_numpy(dtype=float)
).all()

assert (
    actual_df["ACTUAL_DURATION"] > 0
).all()


# ------------------------------------------------------------
# Original cohort must exist in final test set
# ------------------------------------------------------------

missing_original = (
    set(original_ids)
    -
    set(actual_df["LOG_ID"])
)


assert not missing_original, (
    "Original frozen cohort contains IDs not present "
    "in the final test cohort:\n"
    f"{sorted(missing_original)}"
)


print("=" * 75)
print("BASE DATA AUDIT: PASS")
print("=" * 75)

print(
    f"Final held-out test cases : "
    f"{len(actual_df)}"
)

print(
    f"Original frozen cases     : "
    f"{len(original_ids)}"
)

print(
    f"Actual duration range     : "
    f"{actual_df['ACTUAL_DURATION'].min():.2f}"
    f" – "
    f"{actual_df['ACTUAL_DURATION'].max():.2f} min"
)

In [ ]:
# ============================================================
# 3. LOAD AND VERIFY FINAL MODEL-SPECIFIC OPTIMIZER INPUTS
# ============================================================

full_model_data = {}


for model_name, model_path in MODELS.items():

    print("\n" + "-" * 75)
    print(f"Checking {model_name}")
    print("-" * 75)

    assert os.path.exists(
        model_path
    ), (
        f"Missing prescriptive input: "
        f"{model_path}"
    )


    df = pd.read_csv(
        model_path
    )


    required_columns = {
        "LOG_ID",
        "DURATION_P50_MINS",
        "DURATION_P90_MINS"
    }


    missing_columns = (
        required_columns
        -
        set(df.columns)
    )


    assert not missing_columns, (
        f"{model_name}: missing columns "
        f"{missing_columns}"
    )


    df = df.copy()

    df["LOG_ID"] = (
        df["LOG_ID"]
        .astype(str)
    )


    # --------------------------------------------------------
    # ID integrity
    # --------------------------------------------------------

    assert df["LOG_ID"].notna().all()

    assert df["LOG_ID"].is_unique


    # --------------------------------------------------------
    # Prediction integrity
    # --------------------------------------------------------

    pred_cols = [
        "DURATION_P50_MINS",
        "DURATION_P90_MINS"
    ]


    assert df[pred_cols].notna().all().all()

    assert np.isfinite(
        df[pred_cols]
        .to_numpy(dtype=float)
    ).all()


    assert (
        df["DURATION_P50_MINS"] > 0
    ).all()


    assert (
        df["DURATION_P90_MINS"]
        >=
        df["DURATION_P50_MINS"]
    ).all()


    full_model_data[
        model_name
    ] = df


    print(
        f"{model_name}: PASS "
        f"| shape={df.shape}"
    )

In [ ]:
# ============================================================
# 4. CONSTRUCT COMMON ELIGIBLE SAMPLING POOL
#
# Eligibility:
#   1. Present in final temporally held-out test cohort
#   2. Actual duration available
#   3. Present in ALL three final model prediction files
#   4. Excludes original fixed cohort
#
# IMPORTANT:
# No filtering by:
#   - predicted duration
#   - actual duration magnitude
#   - uncertainty width
#   - scheduling performance
#   - workload
#
# Therefore cohort selection remains independent of model
# predictions and realised scheduling outcomes.
# ============================================================

eligible_ids = set(
    actual_df["LOG_ID"]
)


for model_name, df in full_model_data.items():

    eligible_ids = (
        eligible_ids
        &
        set(df["LOG_ID"])
    )


# ------------------------------------------------------------
# Exclude original frozen cohort
# ------------------------------------------------------------

eligible_ids = (
    eligible_ids
    -
    set(original_ids)
)


eligible_ids = sorted(
    eligible_ids
)


assert len(eligible_ids) >= (
    N_CASES
    *
    N_ADDITIONAL_COHORTS
), (
    "Insufficient eligible cases for "
    "two additional cohorts."
)


# ------------------------------------------------------------
# Cross-model alignment check
# ------------------------------------------------------------

for model_name, df in full_model_data.items():

    missing = (
        set(eligible_ids)
        -
        set(df["LOG_ID"])
    )

    assert not missing


print("=" * 75)
print("COMMON ELIGIBLE SAMPLING POOL: PASS")
print("=" * 75)

print(
    f"Final test cases             : "
    f"{len(actual_df)}"
)

print(
    f"Original cohort excluded     : "
    f"{len(original_ids)}"
)

print(
    f"Common eligible cases        : "
    f"{len(eligible_ids)}"
)

print(
    "Sampling criteria do NOT depend on "
    "prediction magnitude or realised performance."
)

In [ ]:
# ============================================================
# 5. FIXED-SEED RANDOM COHORT GENERATION
#
# A single random permutation is generated using seed 42.
#
# First 12 eligible cases  -> Additional Cohort A
# Next 12 eligible cases   -> Additional Cohort B
#
# This guarantees:
#   - reproducibility
#   - no overlap with original cohort
#   - no overlap between A and B
#   - no outcome-based reselection
# ============================================================

rng = np.random.default_rng(
    RANDOM_SEED
)


eligible_array = np.array(
    eligible_ids,
    dtype=object
)


permuted_ids = rng.permutation(
    eligible_array
)


cohort_A_ids = (
    permuted_ids[
        :N_CASES
    ]
    .tolist()
)


cohort_B_ids = (
    permuted_ids[
        N_CASES:
        2 * N_CASES
    ]
    .tolist()
)


# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

assert len(cohort_A_ids) == N_CASES

assert len(cohort_B_ids) == N_CASES

assert len(set(cohort_A_ids)) == N_CASES

assert len(set(cohort_B_ids)) == N_CASES


assert (
    set(cohort_A_ids)
    .isdisjoint(
        set(cohort_B_ids)
    )
), (
    "Additional cohorts A and B overlap."
)


assert (
    set(cohort_A_ids)
    .isdisjoint(
        set(original_ids)
    )
), (
    "Additional cohort A overlaps original cohort."
)


assert (
    set(cohort_B_ids)
    .isdisjoint(
        set(original_ids)
    )
), (
    "Additional cohort B overlaps original cohort."
)


print("=" * 75)
print("FIXED-SEED COHORT SAMPLING: PASS")
print("=" * 75)


print("\nAdditional Cohort A")

for i, log_id in enumerate(
    cohort_A_ids,
    start=1
):

    print(
        f"{i:02d}: {log_id}"
    )


print("\nAdditional Cohort B")

for i, log_id in enumerate(
    cohort_B_ids,
    start=1
):

    print(
        f"{i:02d}: {log_id}"
    )

In [ ]:
# ============================================================
# 6. FREEZE ADDITIONAL COHORT DEFINITIONS
# ============================================================

cohort_A_df = pd.DataFrame(
    {
        "LOG_ID":
            cohort_A_ids,

        "Cohort":
            "Additional_Cohort_A",

        "Sampling_Seed":
            RANDOM_SEED,

        "Sampling_Method":
            "fixed_seed_random_without_replacement"
    }
)


cohort_B_df = pd.DataFrame(
    {
        "LOG_ID":
            cohort_B_ids,

        "Cohort":
            "Additional_Cohort_B",

        "Sampling_Seed":
            RANDOM_SEED,

        "Sampling_Method":
            "fixed_seed_random_without_replacement"
    }
)


cohort_A_df.to_csv(
    COHORT_A_PATH,
    index=False
)


cohort_B_df.to_csv(
    COHORT_B_PATH,
    index=False
)


print("=" * 75)
print("ADDITIONAL COHORTS FROZEN")
print("=" * 75)

print(
    f"Cohort A saved: "
    f"{COHORT_A_PATH}"
)

print(
    f"Cohort B saved: "
    f"{COHORT_B_PATH}"
)

In [ ]:
# ============================================================
# 7. POST-SELECTION DESCRIPTIVE WORKLOAD AUDIT
#
# IMPORTANT:
# Actual workload is inspected ONLY AFTER random cohort
# selection has been completed and frozen.
#
# No cohort will be re-selected based on these values.
# ============================================================

COHORTS = {

    "Additional_Cohort_A":
        cohort_A_ids,

    "Additional_Cohort_B":
        cohort_B_ids
}


actual_duration_map = dict(
    zip(
        actual_df["LOG_ID"],
        actual_df["ACTUAL_DURATION"]
        .astype(float)
    )
)


cohort_audit_rows = []


for cohort_name, cohort_ids in COHORTS.items():

    durations = np.array(
        [
            actual_duration_map[
                log_id
            ]
            for log_id in cohort_ids
        ],
        dtype=float
    )


    total_actual = float(
        durations.sum()
    )


    nominal_capacity = float(
        N_ROOMS
        *
        ROOM_CAPACITY
    )


    load_ratio = (
        total_actual
        /
        nominal_capacity
    )


    cohort_audit_rows.append({

        "Cohort":
            cohort_name,

        "N_Cases":
            len(cohort_ids),

        "Total_Actual_Duration":
            total_actual,

        "Mean_Actual_Duration":
            float(
                durations.mean()
            ),

        "Median_Actual_Duration":
            float(
                np.median(
                    durations
                )
            ),

        "Min_Actual_Duration":
            float(
                durations.min()
            ),

        "Max_Actual_Duration":
            float(
                durations.max()
            ),

        "Nominal_Total_Capacity":
            nominal_capacity,

        "Actual_Capacity_Load_Ratio":
            load_ratio
    })


cohort_audit_df = pd.DataFrame(
    cohort_audit_rows
)


print("=" * 75)
print("POST-SELECTION COHORT WORKLOAD AUDIT")
print("=" * 75)

display(
    cohort_audit_df
)


# ------------------------------------------------------------
# Explicit non-overlap audit
# ------------------------------------------------------------

print("\nOverlap audit:")

print(
    "Original ∩ A:",
    len(
        set(original_ids)
        &
        set(cohort_A_ids)
    )
)

print(
    "Original ∩ B:",
    len(
        set(original_ids)
        &
        set(cohort_B_ids)
    )
)

print(
    "A ∩ B:",
    len(
        set(cohort_A_ids)
        &
        set(cohort_B_ids)
    )
)


assert len(
    set(original_ids)
    &
    set(cohort_A_ids)
) == 0

assert len(
    set(original_ids)
    &
    set(cohort_B_ids)
) == 0

assert len(
    set(cohort_A_ids)
    &
    set(cohort_B_ids)
) == 0


print("\nCOHORT NON-OVERLAP AUDIT: PASS")

In [ ]:
# ============================================================
# 8. PREPARE MODEL × COHORT OPTIMIZER INPUTS
# ============================================================

prepared_inputs = {}


for cohort_name, cohort_ids in COHORTS.items():

    prepared_inputs[
        cohort_name
    ] = {}


    print("\n" + "=" * 75)

    print(
        f"PREPARING {cohort_name}"
    )

    print("=" * 75)


    for model_name, full_df in full_model_data.items():

        df_day = (
            full_df
            .set_index("LOG_ID")
            .loc[cohort_ids]
            .reset_index()
        )


        assert len(df_day) == N_CASES


        assert (
            df_day["LOG_ID"]
            .tolist()
            ==
            cohort_ids
        )


        assert (
            df_day[
                "DURATION_P50_MINS"
            ]
            .notna()
            .all()
        )


        assert (
            df_day[
                "DURATION_P90_MINS"
            ]
            .notna()
            .all()
        )


        prepared_inputs[
            cohort_name
        ][
            model_name
        ] = df_day


        print(
            f"{model_name:10s}: "
            f"{df_day.shape}"
        )


# ------------------------------------------------------------
# Cross-model ID alignment
# ------------------------------------------------------------

for cohort_name in COHORTS:

    reference = (
        prepared_inputs[
            cohort_name
        ][
            "LightGBM"
        ][
            "LOG_ID"
        ]
        .tolist()
    )


    for model_name in MODELS:

        current = (
            prepared_inputs[
                cohort_name
            ][
                model_name
            ][
                "LOG_ID"
            ]
            .tolist()
        )


        assert (
            current
            ==
            reference
        )


print("\n" + "=" * 75)
print("MODEL × COHORT INPUT ALIGNMENT: PASS")
print("=" * 75)

In [ ]:
# ============================================================
# 9. REALISED SCHEDULE REPLAY
#
# No hindsight re-optimisation.
#
# Room assignment and planned within-room order are frozen
# from the MILP incumbent.
#
# Actual duration is introduced only after optimisation.
# ============================================================


def replay_realised_schedule(
    df_day,
    milp_result,
    actual_duration_map,
    model_name,
    lam,
    cohort_name,
    n_rooms=3,
    room_capacity=480,
    turnover=20,
    duration_cap=360.0
):

    # --------------------------------------------------------
    # Extract planned MILP schedule
    # --------------------------------------------------------

    room_assignment = (
        milp_result[
            "room_assignment"
        ]
    )

    start_times = (
        milp_result[
            "start_times"
        ]
    )


    assert len(
        room_assignment
    ) == len(df_day)

    assert len(
        start_times
    ) == len(df_day)


    # --------------------------------------------------------
    # Reconstruct planning durations
    # --------------------------------------------------------

    p50 = (
        df_day[
            "DURATION_P50_MINS"
        ]
        .to_numpy(
            dtype=float
        )
    )


    p90 = (
        df_day[
            "DURATION_P90_MINS"
        ]
        .to_numpy(
            dtype=float
        )
    )


    planning_uncapped = (
        p50
        +
        float(lam)
        *
        (
            p90
            -
            p50
        )
    )


    if duration_cap is None:

        planning_duration = (
            planning_uncapped.copy()
        )

        capped_mask = np.zeros(
            len(df_day),
            dtype=bool
        )

    else:

        capped_mask = (
            planning_uncapped
            >
            float(
                duration_cap
            )
        )

        planning_duration = np.minimum(
            planning_uncapped,
            float(
                duration_cap
            )
        )


    # --------------------------------------------------------
    # Build schedule table
    # --------------------------------------------------------

    rows = []


    for i in range(
        len(df_day)
    ):

        log_id = str(
            df_day.iloc[i][
                "LOG_ID"
            ]
        )


        assert (
            log_id
            in
            actual_duration_map
        )


        rows.append({

            "Cohort":
                cohort_name,

            "Model":
                model_name,

            "Lambda":
                float(lam),

            "Case_Index":
                int(i),

            "LOG_ID":
                log_id,

            "Room":
                int(
                    room_assignment[i]
                ),

            "Planned_Start":
                float(
                    start_times[i]
                ),

            "P50_Duration":
                float(
                    p50[i]
                ),

            "P90_Duration":
                float(
                    p90[i]
                ),

            "Planning_Duration_Uncapped":
                float(
                    planning_uncapped[i]
                ),

            "Planning_Duration":
                float(
                    planning_duration[i]
                ),

            "Duration_Was_Capped":
                bool(
                    capped_mask[i]
                ),

            "Actual_Duration":
                float(
                    actual_duration_map[
                        log_id
                    ]
                )
        })


    schedule_df = pd.DataFrame(
        rows
    )


    # --------------------------------------------------------
    # Recover planned within-room sequence
    # --------------------------------------------------------

    schedule_df = (
        schedule_df
        .sort_values(
            [
                "Room",
                "Planned_Start",
                "Case_Index"
            ]
        )
        .reset_index(
            drop=True
        )
    )


    schedule_df[
        "Sequence_Position"
    ] = (
        schedule_df
        .groupby(
            "Room"
        )
        .cumcount()
        +
        1
    )


    # --------------------------------------------------------
    # Initialise realised columns
    # --------------------------------------------------------

    schedule_df[
        "Realised_Start"
    ] = np.nan

    schedule_df[
        "Realised_End"
    ] = np.nan

    schedule_df[
        "Start_Delay"
    ] = np.nan


    # --------------------------------------------------------
    # Replay each room independently
    # --------------------------------------------------------

    for room in range(
        n_rooms
    ):

        room_indices = (
            schedule_df.index[
                schedule_df[
                    "Room"
                ]
                ==
                room
            ]
            .tolist()
        )


        previous_end = None


        for row_idx in room_indices:

            planned_start = float(
                schedule_df.at[
                    row_idx,
                    "Planned_Start"
                ]
            )


            actual_duration = float(
                schedule_df.at[
                    row_idx,
                    "Actual_Duration"
                ]
            )


            if previous_end is None:

                realised_start = (
                    planned_start
                )

            else:

                realised_start = max(
                    planned_start,
                    previous_end
                    +
                    turnover
                )


            realised_end = (
                realised_start
                +
                actual_duration
            )


            start_delay = (
                realised_start
                -
                planned_start
            )


            schedule_df.at[
                row_idx,
                "Realised_Start"
            ] = realised_start


            schedule_df.at[
                row_idx,
                "Realised_End"
            ] = realised_end


            schedule_df.at[
                row_idx,
                "Start_Delay"
            ] = start_delay


            previous_end = (
                realised_end
            )


    # --------------------------------------------------------
    # Basic replay integrity
    # --------------------------------------------------------

    assert (
        schedule_df[
            "Realised_Start"
        ]
        >=
        schedule_df[
            "Planned_Start"
        ]
        -
        1e-7
    ).all()


    assert (
        schedule_df[
            "Start_Delay"
        ]
        >=
        -1e-7
    ).all()


    # --------------------------------------------------------
    # Explicit turnover audit
    # --------------------------------------------------------

    turnover_violations = []


    for room in range(
        n_rooms
    ):

        room_schedule = (
            schedule_df[
                schedule_df[
                    "Room"
                ]
                ==
                room
            ]
            .sort_values(
                "Sequence_Position"
            )
            .reset_index(
                drop=True
            )
        )


        for j in range(
            1,
            len(
                room_schedule
            )
        ):

            previous_end = float(
                room_schedule.loc[
                    j - 1,
                    "Realised_End"
                ]
            )


            current_start = float(
                room_schedule.loc[
                    j,
                    "Realised_Start"
                ]
            )


            required_start = (
                previous_end
                +
                turnover
            )


            if (
                current_start
                <
                required_start
                -
                1e-6
            ):

                turnover_violations.append({

                    "Room":
                        room,

                    "Previous_LOG_ID":
                        room_schedule.loc[
                            j - 1,
                            "LOG_ID"
                        ],

                    "Current_LOG_ID":
                        room_schedule.loc[
                            j,
                            "LOG_ID"
                        ],

                    "Required_Start":
                        required_start,

                    "Actual_Start":
                        current_start
                })


    assert (
        len(
            turnover_violations
        )
        ==
        0
    ), (
        "Turnover violation detected:\n"
        f"{turnover_violations}"
    )


    # --------------------------------------------------------
    # Room-level metrics
    # --------------------------------------------------------

    room_rows = []


    for room in range(
        n_rooms
    ):

        r = (
            schedule_df[
                schedule_df[
                    "Room"
                ]
                ==
                room
            ]
            .sort_values(
                "Sequence_Position"
            )
        )


        if len(r) == 0:

            n_room_cases = 0

            actual_workload = 0.0

            realised_finish = 0.0

            total_delay = 0.0

            mean_delay = 0.0

            max_delay = 0.0

        else:

            n_room_cases = int(
                len(r)
            )


            actual_workload = float(
                r[
                    "Actual_Duration"
                ]
                .sum()
            )


            realised_finish = float(
                r[
                    "Realised_End"
                ]
                .max()
            )


            total_delay = float(
                r[
                    "Start_Delay"
                ]
                .sum()
            )


            mean_delay = float(
                r[
                    "Start_Delay"
                ]
                .mean()
            )


            max_delay = float(
                r[
                    "Start_Delay"
                ]
                .max()
            )


        realised_overtime = max(
            realised_finish
            -
            room_capacity,
            0.0
        )


        room_rows.append({

            "Cohort":
                cohort_name,

            "Model":
                model_name,

            "Lambda":
                float(lam),

            "Room":
                int(room),

            "N_Cases":
                n_room_cases,

            "Actual_Surgical_Workload":
                actual_workload,

            "Realised_Room_Finish":
                realised_finish,

            "Realised_Overtime":
                realised_overtime,

            "Total_Start_Delay":
                total_delay,

            "Mean_Start_Delay":
                mean_delay,

            "Max_Start_Delay":
                max_delay
        })


    room_df = pd.DataFrame(
        room_rows
    )


    # --------------------------------------------------------
    # System-level realised metrics
    # --------------------------------------------------------

    realised_makespan = float(
        room_df[
            "Realised_Room_Finish"
        ]
        .max()
    )


    realised_overtime = float(
        room_df[
            "Realised_Overtime"
        ]
        .sum()
    )


    total_start_delay = float(
        schedule_df[
            "Start_Delay"
        ]
        .sum()
    )


    mean_start_delay = float(
        schedule_df[
            "Start_Delay"
        ]
        .mean()
    )


    max_start_delay = float(
        schedule_df[
            "Start_Delay"
        ]
        .max()
    )


    delayed_cases = int(
        (
            schedule_df[
                "Start_Delay"
            ]
            >
            1e-6
        )
        .sum()
    )


    actual_total_duration = float(
        schedule_df[
            "Actual_Duration"
        ]
        .sum()
    )


    nominal_capacity = float(
        n_rooms
        *
        room_capacity
    )


    actual_load_ratio = (
        actual_total_duration
        /
        nominal_capacity
    )


    # --------------------------------------------------------
    # Descriptive evaluation score only.
    #
    # NOT the original MILP objective.
    # --------------------------------------------------------

    realised_evaluation_score = (
        realised_overtime
        +
        0.5
        *
        realised_makespan
    )


    summary = {

        "Cohort":
            cohort_name,

        "Model":
            model_name,

        "Lambda":
            float(lam),

        "N_Cases":
            int(
                len(
                    schedule_df
                )
            ),

        "Actual_Total_Duration":
            actual_total_duration,

        "Nominal_Total_Capacity":
            nominal_capacity,

        "Actual_Capacity_Load_Ratio":
            actual_load_ratio,

        "Realised_Makespan":
            realised_makespan,

        "Realised_Overtime":
            realised_overtime,

        "Total_Start_Delay":
            total_start_delay,

        "Mean_Start_Delay":
            mean_start_delay,

        "Max_Start_Delay":
            max_start_delay,

        "Delayed_Cases":
            delayed_cases,

        "Delayed_Case_Rate":
            (
                delayed_cases
                /
                len(
                    schedule_df
                )
            ),

        "Realised_Evaluation_Score":
            realised_evaluation_score,

        "Replay_Turnover_Violations":
            len(
                turnover_violations
            ),

        # ----------------------------------------------------
        # MILP diagnostics
        # ----------------------------------------------------

        "MILP_Objective":
            float(
                milp_result[
                    "Objective"
                ]
            ),

        "MILP_Planned_Overtime":
            float(
                milp_result[
                    "Overtime"
                ]
            ),

        "MILP_Planned_Makespan":
            float(
                milp_result[
                    "Makespan"
                ]
            ),

        "MILP_Gap":
            float(
                milp_result[
                    "Gap"
                ]
            ),

        "MILP_Solve_Time":
            float(
                milp_result[
                    "Solve_Time"
                ]
            ),

        "MILP_Status":
            int(
                milp_result[
                    "Status"
                ]
            ),

        "MILP_Hit_Time_Limit":
            bool(
                milp_result[
                    "Hit_Time_Limit"
                ]
            ),

        "MILP_Reached_1pct_Gap":
            bool(
                milp_result[
                    "Reached_1pct_Gap"
                ]
            ),

        "MILP_N_Capped":
            int(
                milp_result[
                    "N_Capped"
                ]
            ),

        "MILP_Pct_Capped":
            float(
                milp_result[
                    "Pct_Capped"
                ]
            ),

        "MILP_Max_Uncapped_Duration":
            float(
                milp_result[
                    "Max_Uncapped_Duration"
                ]
            ),

        "MILP_Max_Planning_Duration":
            float(
                milp_result[
                    "Max_Planning_Duration"
                ]
            )
    }


    return (
        schedule_df,
        room_df,
        summary
    )

In [ ]:
# ============================================================
# 10. ADDITIONAL COHORT RUN PLAN
# ============================================================

RUN_PLAN = [

    (
        "Additional_Cohort_A",
        "LightGBM",
        VALIDATION_LAMBDA
    ),

    (
        "Additional_Cohort_A",
        "RF",
        VALIDATION_LAMBDA
    ),

    (
        "Additional_Cohort_A",
        "XGBoost",
        VALIDATION_LAMBDA
    ),

    (
        "Additional_Cohort_B",
        "LightGBM",
        VALIDATION_LAMBDA
    ),

    (
        "Additional_Cohort_B",
        "RF",
        VALIDATION_LAMBDA
    ),

    (
        "Additional_Cohort_B",
        "XGBoost",
        VALIDATION_LAMBDA
    )
]


print("=" * 75)
print("ADDITIONAL COHORT RUN PLAN")
print("=" * 75)


for i, (
    cohort_name,
    model_name,
    lam
) in enumerate(
    RUN_PLAN,
    start=1
):

    print(
        f"{i}: "
        f"{cohort_name} | "
        f"{model_name} | "
        f"lambda={lam}"
    )


print(
    f"\nTotal planned runs: "
    f"{len(RUN_PLAN)}"
)

In [ ]:
# ============================================================
# 11. RUN ADDITIONAL COHORT VALIDATION
#
# Checkpoint/resume enabled.
#
# Each successful MILP incumbent is immediately:
#   1. replayed using actual durations
#   2. saved at case level
#   3. saved at room level
#   4. saved at run level
#   5. checkpointed
# ============================================================


# ------------------------------------------------------------
# Load checkpoint
# ------------------------------------------------------------

if os.path.exists(
    CHECKPOINT_OUTPUT
):

    checkpoint_df = pd.read_csv(
        CHECKPOINT_OUTPUT
    )

    print(
        f"Existing checkpoint loaded: "
        f"{len(checkpoint_df)} rows."
    )

else:

    checkpoint_df = pd.DataFrame()

    print(
        "No existing checkpoint found."
    )


# ------------------------------------------------------------
# Completed run keys
# ------------------------------------------------------------

completed_keys = set()


if len(
    checkpoint_df
) > 0:

    required_cols = {
        "Cohort",
        "Model",
        "Lambda"
    }


    assert required_cols.issubset(
        checkpoint_df.columns
    )


    for _, row in checkpoint_df.iterrows():

        completed_keys.add(

            (
                str(
                    row[
                        "Cohort"
                    ]
                ),

                str(
                    row[
                        "Model"
                    ]
                ),

                round(
                    float(
                        row[
                            "Lambda"
                        ]
                    ),
                    6
                )
            )
        )


print(
    f"Completed runs: "
    f"{completed_keys}"
)


# ------------------------------------------------------------
# Existing detailed outputs
# ------------------------------------------------------------

if os.path.exists(
    SCHEDULE_OUTPUT
):

    all_schedule_df = pd.read_csv(
        SCHEDULE_OUTPUT
    )

else:

    all_schedule_df = pd.DataFrame()


if os.path.exists(
    ROOM_OUTPUT
):

    all_room_df = pd.read_csv(
        ROOM_OUTPUT
    )

else:

    all_room_df = pd.DataFrame()


if os.path.exists(
    SUMMARY_OUTPUT
):

    all_summary_df = pd.read_csv(
        SUMMARY_OUTPUT
    )

else:

    all_summary_df = pd.DataFrame()


# ------------------------------------------------------------
# Run loop
# ------------------------------------------------------------

experiment_start = time.time()


for run_number, (
    cohort_name,
    model_name,
    lam
) in enumerate(
    RUN_PLAN,
    start=1
):

    lam = float(
        lam
    )


    run_key = (

        cohort_name,

        model_name,

        round(
            lam,
            6
        )
    )


    print("\n" + "=" * 75)

    print(
        f"RUN "
        f"{run_number}/"
        f"{len(RUN_PLAN)}"
    )

    print(
        f"Cohort : "
        f"{cohort_name}"
    )

    print(
        f"Model  : "
        f"{model_name}"
    )

    print(
        f"Lambda : "
        f"{lam}"
    )

    print("=" * 75)


    # --------------------------------------------------------
    # Resume protection
    # --------------------------------------------------------

    if (
        run_key
        in
        completed_keys
    ):

        print(
            "SKIPPED — already completed."
        )

        continue


    df_day = (
        prepared_inputs[
            cohort_name
        ][
            model_name
        ]
        .copy()
    )


    run_start = time.time()


    try:

        # ====================================================
        # SOLVE MILP
        # ====================================================

        result = solve_or_milp(

            df_day=df_day,

            lam=lam,

            n_rooms=N_ROOMS,

            room_capacity=ROOM_CAPACITY,

            turnover=TURNOVER,

            balance_weight=BALANCE_WEIGHT,

            duration_cap=DURATION_CAP,

            time_limit=TIME_LIMIT
        )


        if result is None:

            raise RuntimeError(
                "MILP returned no feasible incumbent."
            )


        # ====================================================
        # REALISED REPLAY
        # ====================================================

        (
            schedule_df,
            room_df,
            summary
        ) = replay_realised_schedule(

            df_day=df_day,

            milp_result=result,

            actual_duration_map=actual_duration_map,

            model_name=model_name,

            lam=lam,

            cohort_name=cohort_name,

            n_rooms=N_ROOMS,

            room_capacity=ROOM_CAPACITY,

            turnover=TURNOVER,

            duration_cap=DURATION_CAP
        )


        # ----------------------------------------------------
        # Validation flag
        # ----------------------------------------------------

        summary[
            "Validation_Run_Success"
        ] = True

        summary[
            "Failure_Type"
        ] = ""

        summary[
            "Failure_Message"
        ] = ""


        # ====================================================
        # APPEND CASE-LEVEL OUTPUT
        # ====================================================

        if len(
            all_schedule_df
        ) == 0:

            all_schedule_df = (
                schedule_df.copy()
            )

        else:

            all_schedule_df = pd.concat(
                [
                    all_schedule_df,
                    schedule_df
                ],
                ignore_index=True
            )


        # ====================================================
        # APPEND ROOM-LEVEL OUTPUT
        # ====================================================

        if len(
            all_room_df
        ) == 0:

            all_room_df = (
                room_df.copy()
            )

        else:

            all_room_df = pd.concat(
                [
                    all_room_df,
                    room_df
                ],
                ignore_index=True
            )


        # ====================================================
        # APPEND RUN-LEVEL OUTPUT
        # ====================================================

        summary_row = pd.DataFrame(
            [
                summary
            ]
        )


        if len(
            all_summary_df
        ) == 0:

            all_summary_df = (
                summary_row.copy()
            )

        else:

            all_summary_df = pd.concat(
                [
                    all_summary_df,
                    summary_row
                ],
                ignore_index=True
            )


        # ====================================================
        # SAVE ALL OUTPUTS IMMEDIATELY
        # ====================================================

        all_schedule_df.to_csv(
            SCHEDULE_OUTPUT,
            index=False
        )


        all_room_df.to_csv(
            ROOM_OUTPUT,
            index=False
        )


        all_summary_df.to_csv(
            SUMMARY_OUTPUT,
            index=False
        )


        # ====================================================
        # CHECKPOINT
        # ====================================================

        checkpoint_row = pd.DataFrame(
            [
                {
                    "Cohort":
                        cohort_name,

                    "Model":
                        model_name,

                    "Lambda":
                        lam,

                    "Completed":
                        True,

                    "Validation_Run_Success":
                        True,

                    "MILP_Gap":
                        summary[
                            "MILP_Gap"
                        ],

                    "MILP_Solve_Time":
                        summary[
                            "MILP_Solve_Time"
                        ],

                    "Realised_Makespan":
                        summary[
                            "Realised_Makespan"
                        ],

                    "Realised_Overtime":
                        summary[
                            "Realised_Overtime"
                        ],

                    "Total_Start_Delay":
                        summary[
                            "Total_Start_Delay"
                        ]
                }
            ]
        )


        if len(
            checkpoint_df
        ) == 0:

            checkpoint_df = (
                checkpoint_row.copy()
            )

        else:

            checkpoint_df = pd.concat(
                [
                    checkpoint_df,
                    checkpoint_row
                ],
                ignore_index=True
            )


        checkpoint_df.to_csv(
            CHECKPOINT_OUTPUT,
            index=False
        )


        completed_keys.add(
            run_key
        )


        # ====================================================
        # CONSOLE OUTPUT
        # ====================================================

        elapsed = (
            time.time()
            -
            run_start
        )


        print("\nRUN COMPLETE")

        print(
            f"MILP objective        : "
            f"{summary['MILP_Objective']:.2f}"
        )

        print(
            f"MILP solve time       : "
            f"{summary['MILP_Solve_Time']:.2f} s"
        )

        print(
            f"MILP gap              : "
            f"{100 * summary['MILP_Gap']:.2f}%"
        )

        print(
            f"Hit time limit        : "
            f"{summary['MILP_Hit_Time_Limit']}"
        )

        print(
            f"Capped cases          : "
            f"{summary['MILP_N_Capped']}"
        )

        print(
            f"Actual load ratio     : "
            f"{summary['Actual_Capacity_Load_Ratio']:.4f}"
        )

        print(
            f"Realised makespan     : "
            f"{summary['Realised_Makespan']:.2f}"
        )

        print(
            f"Realised overtime     : "
            f"{summary['Realised_Overtime']:.2f}"
        )

        print(
            f"Total start delay     : "
            f"{summary['Total_Start_Delay']:.2f}"
        )

        print(
            f"Delayed cases         : "
            f"{summary['Delayed_Cases']}"
        )

        print(
            f"Turnover violations   : "
            f"{summary['Replay_Turnover_Violations']}"
        )

        print(
            f"Wall-clock run        : "
            f"{elapsed:.2f} s"
        )


    except Exception as exc:

        print("\nRUN FAILED")

        print(
            f"{type(exc).__name__}: "
            f"{exc}"
        )

        raise


print("\n" + "=" * 75)
print("ADDITIONAL COHORT RUN LOOP FINISHED")
print("=" * 75)

print(
    f"Elapsed time: "
    f"{(time.time() - experiment_start) / 60:.2f} min"
)

In [ ]:
# ============================================================
# 12. FINAL ADDITIONAL-COHORT INTEGRITY AUDIT
# ============================================================

summary_df = pd.read_csv(
    SUMMARY_OUTPUT
)

schedule_df = pd.read_csv(
    SCHEDULE_OUTPUT
)

room_df = pd.read_csv(
    ROOM_OUTPUT
)


# ------------------------------------------------------------
# Defensive duplicate removal
# ------------------------------------------------------------

summary_df = (
    summary_df
    .drop_duplicates(
        subset=[
            "Cohort",
            "Model",
            "Lambda"
        ],
        keep="last"
    )
    .reset_index(
        drop=True
    )
)


schedule_df = (
    schedule_df
    .drop_duplicates(
        subset=[
            "Cohort",
            "Model",
            "Lambda",
            "LOG_ID"
        ],
        keep="last"
    )
    .reset_index(
        drop=True
    )
)


room_df = (
    room_df
    .drop_duplicates(
        subset=[
            "Cohort",
            "Model",
            "Lambda",
            "Room"
        ],
        keep="last"
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Expected six runs
# ------------------------------------------------------------

expected_keys = set(

    (
        cohort_name,
        model_name,
        float(lam)
    )

    for (
        cohort_name,
        model_name,
        lam
    )

    in RUN_PLAN
)


actual_keys = set(

    zip(

        summary_df[
            "Cohort"
        ],

        summary_df[
            "Model"
        ],

        summary_df[
            "Lambda"
        ]
        .astype(float)
    )
)


missing_keys = (
    expected_keys
    -
    actual_keys
)


assert not missing_keys, (
    "Missing validation runs:\n"
    f"{sorted(missing_keys)}"
)


# ------------------------------------------------------------
# Exactly 12 unique cases per run
# ------------------------------------------------------------

case_counts = (

    schedule_df
    .groupby(
        [
            "Cohort",
            "Model",
            "Lambda"
        ]
    )[
        "LOG_ID"
    ]
    .nunique()
)


assert (
    case_counts
    ==
    N_CASES
).all(), (
    "At least one run does not contain "
    "exactly 12 unique cases."
)


# ------------------------------------------------------------
# Exactly 3 rooms per run
# ------------------------------------------------------------

room_counts = (

    room_df
    .groupby(
        [
            "Cohort",
            "Model",
            "Lambda"
        ]
    )[
        "Room"
    ]
    .nunique()
)


assert (
    room_counts
    ==
    N_ROOMS
).all(), (
    "At least one run does not contain "
    "exactly three room summaries."
)


# ------------------------------------------------------------
# No turnover violations
# ------------------------------------------------------------

assert (
    summary_df[
        "Replay_Turnover_Violations"
    ]
    ==
    0
).all()


# ------------------------------------------------------------
# Same actual workload across models within each cohort
# ------------------------------------------------------------

for cohort_name in COHORTS:

    cohort_summary = (
        summary_df[
            summary_df[
                "Cohort"
            ]
            ==
            cohort_name
        ]
    )


    actual_totals = (
        cohort_summary[
            "Actual_Total_Duration"
        ]
        .to_numpy(
            dtype=float
        )
    )


    assert len(
        actual_totals
    ) == len(
        MODELS
    )


    assert np.allclose(
        actual_totals,
        actual_totals[0]
    )


# ------------------------------------------------------------
# Cross-cohort non-overlap remains intact
# ------------------------------------------------------------

A_ids_saved = set(
    pd.read_csv(
        COHORT_A_PATH
    )[
        "LOG_ID"
    ]
    .astype(str)
)


B_ids_saved = set(
    pd.read_csv(
        COHORT_B_PATH
    )[
        "LOG_ID"
    ]
    .astype(str)
)


assert (
    A_ids_saved
    .isdisjoint(
        B_ids_saved
    )
)


assert (
    A_ids_saved
    .isdisjoint(
        set(
            original_ids
        )
    )
)


assert (
    B_ids_saved
    .isdisjoint(
        set(
            original_ids
        )
    )
)


# ------------------------------------------------------------
# Save cleaned outputs
# ------------------------------------------------------------

summary_df.to_csv(
    SUMMARY_OUTPUT,
    index=False
)

schedule_df.to_csv(
    SCHEDULE_OUTPUT,
    index=False
)

room_df.to_csv(
    ROOM_OUTPUT,
    index=False
)


print("=" * 75)
print("ADDITIONAL COHORT FINAL AUDIT: PASS")
print("=" * 75)

print(
    f"Successful runs       : "
    f"{len(expected_keys)}"
)

print(
    f"Case-level rows       : "
    f"{len(schedule_df)}"
)

print(
    f"Room-level rows       : "
    f"{len(room_df)}"
)

print(
    "Turnover violations  : 0"
)

print(
    "Original/A/B overlap : 0"
)

In [ ]:
# ============================================================
# 13. ADDITIONAL COHORT COMPARISON TABLE
# ============================================================

comparison_columns = [

    "Cohort",
    "Model",
    "Lambda",

    "Actual_Total_Duration",
    "Actual_Capacity_Load_Ratio",

    "MILP_N_Capped",

    "MILP_Planned_Makespan",
    "MILP_Planned_Overtime",

    "Realised_Makespan",
    "Realised_Overtime",

    "Total_Start_Delay",
    "Mean_Start_Delay",
    "Max_Start_Delay",

    "Delayed_Cases",

    "MILP_Gap",
    "MILP_Hit_Time_Limit"
]


comparison_df = (
    summary_df[
        comparison_columns
    ]
    .copy()
)


cohort_order = {
    "Additional_Cohort_A": 0,
    "Additional_Cohort_B": 1
}


model_order = {
    "LightGBM": 0,
    "RF": 1,
    "XGBoost": 2
}


comparison_df[
    "_Cohort_Order"
] = (
    comparison_df[
        "Cohort"
    ]
    .map(
        cohort_order
    )
)


comparison_df[
    "_Model_Order"
] = (
    comparison_df[
        "Model"
    ]
    .map(
        model_order
    )
)


comparison_df = (

    comparison_df
    .sort_values(
        [
            "_Cohort_Order",
            "_Model_Order"
        ]
    )
    .drop(
        columns=[
            "_Cohort_Order",
            "_Model_Order"
        ]
    )
    .reset_index(
        drop=True
    )
)


print("=" * 75)
print("ADDITIONAL COHORT ROBUSTNESS RESULTS")
print("=" * 75)

display(
    comparison_df
)

In [ ]:
# ============================================================
# 14. COMPACT REPORTING TABLE
# ============================================================

report_table = (

    summary_df[
        [
            "Cohort",
            "Model",
            "Actual_Capacity_Load_Ratio",
            "Realised_Makespan",
            "Realised_Overtime",
            "Total_Start_Delay",
            "Delayed_Cases",
            "MILP_N_Capped",
            "MILP_Gap"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------

report_table[
    "Actual_Capacity_Load_Ratio"
] = (
    report_table[
        "Actual_Capacity_Load_Ratio"
    ]
    .round(3)
)


report_table[
    "Realised_Makespan"
] = (
    report_table[
        "Realised_Makespan"
    ]
    .round(2)
)


report_table[
    "Realised_Overtime"
] = (
    report_table[
        "Realised_Overtime"
    ]
    .round(2)
)


report_table[
    "Total_Start_Delay"
] = (
    report_table[
        "Total_Start_Delay"
    ]
    .round(2)
)


report_table[
    "MILP_Gap"
] = (
    100
    *
    report_table[
        "MILP_Gap"
    ]
).round(2)


report_table[
    "_Cohort_Order"
] = (
    report_table[
        "Cohort"
    ]
    .map(
        cohort_order
    )
)


report_table[
    "_Model_Order"
] = (
    report_table[
        "Model"
    ]
    .map(
        model_order
    )
)


report_table = (

    report_table
    .sort_values(
        [
            "_Cohort_Order",
            "_Model_Order"
        ]
    )
    .drop(
        columns=[
            "_Cohort_Order",
            "_Model_Order"
        ]
    )
    .reset_index(
        drop=True
    )
)


display(
    report_table
)

In [ ]:
# ============================================================
# 15. DESCRIPTIVE CROSS-COHORT MODEL SUMMARY
#
# NOTE:
# This is descriptive only.
# With only three validation cohorts overall, no inferential
# significance claims should be made.
# ============================================================

cross_cohort_summary = (

    summary_df
    .groupby(
        "Model"
    )
    .agg(

        N_Additional_Cohorts=(
            "Cohort",
            "nunique"
        ),

        Mean_Realised_Makespan=(
            "Realised_Makespan",
            "mean"
        ),

        Mean_Realised_Overtime=(
            "Realised_Overtime",
            "mean"
        ),

        Mean_Total_Start_Delay=(
            "Total_Start_Delay",
            "mean"
        ),

        Mean_Delayed_Cases=(
            "Delayed_Cases",
            "mean"
        ),

        Mean_MILP_Gap=(
            "MILP_Gap",
            "mean"
        )
    )
    .reset_index()
)


cross_cohort_summary[
    "Mean_Realised_Makespan"
] = (
    cross_cohort_summary[
        "Mean_Realised_Makespan"
    ]
    .round(2)
)


cross_cohort_summary[
    "Mean_Realised_Overtime"
] = (
    cross_cohort_summary[
        "Mean_Realised_Overtime"
    ]
    .round(2)
)


cross_cohort_summary[
    "Mean_Total_Start_Delay"
] = (
    cross_cohort_summary[
        "Mean_Total_Start_Delay"
    ]
    .round(2)
)


cross_cohort_summary[
    "Mean_Delayed_Cases"
] = (
    cross_cohort_summary[
        "Mean_Delayed_Cases"
    ]
    .round(2)
)


cross_cohort_summary[
    "Mean_MILP_Gap"
] = (
    100
    *
    cross_cohort_summary[
        "Mean_MILP_Gap"
    ]
).round(2)


cross_cohort_summary[
    "_Model_Order"
] = (
    cross_cohort_summary[
        "Model"
    ]
    .map(
        model_order
    )
)


cross_cohort_summary = (

    cross_cohort_summary
    .sort_values(
        "_Model_Order"
    )
    .drop(
        columns=[
            "_Model_Order"
        ]
    )
    .reset_index(
        drop=True
    )
)


print("=" * 75)
print("DESCRIPTIVE MODEL ROBUSTNESS ACROSS ADDITIONAL COHORTS")
print("=" * 75)

display(
    cross_cohort_summary
)

In [ ]:
# ============================================================
# 16. FINAL STEP 4D EXPERIMENTAL FREEZE SUMMARY
# ============================================================

print("=" * 75)
print("STEP 4D — EXPERIMENTAL FREEZE SUMMARY")
print("=" * 75)

print(
    "Sampling method:"
)

print(
    "  Fixed-seed random sampling without replacement"
)

print(
    f"  Seed = {RANDOM_SEED}"
)


print(
    "\nSampling source:"
)

print(
    "  Final temporally held-out test cohort"
)

print(
    "  Common availability across LGB / RF / XGB"
)

print(
    "  Original 12-case cohort excluded"
)


print(
    "\nValidation design:"
)

print(
    "  Additional Cohort A: 12 cases"
)

print(
    "  Additional Cohort B: 12 cases"
)

print(
    "  3 predictive models"
)

print(
    f"  lambda = {VALIDATION_LAMBDA}"
)

print(
    f"  Total additional MILP runs = {len(RUN_PLAN)}"
)


print(
    "\nMILP configuration:"
)

print(
    f"  Rooms = {N_ROOMS}"
)

print(
    f"  Capacity = {ROOM_CAPACITY} min"
)

print(
    f"  Turnover = {TURNOVER} min"
)

print(
    f"  beta = {BALANCE_WEIGHT}"
)

print(
    f"  Duration cap = {DURATION_CAP} min"
)

print(
    f"  Time limit = {TIME_LIMIT} s"
)


print(
    "\nRealised evaluation:"
)

print(
    "  Planned room assignment frozen"
)

print(
    "  Planned within-room order frozen"
)

print(
    "  Actual durations substituted ex post"
)

print(
    "  Downstream delays propagated"
)

print(
    "  No hindsight re-optimisation"
)


print(
    "\nInterpretation constraint:"
)

print(
    "  Results are descriptive robustness checks."
)

print(
    "  Time-limited feasible incumbents must not be "
    "described as proven-optimal schedules."
)


print("\n" + "=" * 75)
print("STEP 4D COMPLETE")
print("=" * 75)